# Patient Splitting

In this notebook the data will be split to ensure that images that belong to the same patient do not appear again in different modelling splits.



In [1]:
from pathlib import Path
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split


PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

OUTPUT_DIR = DATA_DIR / "derived" / 'splits'




In [2]:
full_train = pd.read_csv(DATA_DIR/ "derived" / "indexes" / "full_train_index.csv")
images_per_patient_ft = full_train.groupby("patient_id").size()

In [3]:
images_per_patient_ft.describe()

count    2056.000000
mean       16.111868
std        15.674491
min         2.000000
25%         5.000000
50%        12.000000
75%        22.000000
max       115.000000
dtype: float64

In [4]:
subset_train = pd.read_csv(DATA_DIR/ "derived" / "indexes" / "subset_train_index.csv")
images_per_patient_st = subset_train.groupby("patient_id").size()

In [5]:
images_per_patient_st.describe()

count    422.000000
mean      16.265403
std       16.568455
min        3.000000
25%        5.000000
50%       12.000000
75%       21.750000
max      115.000000
dtype: float64

In [6]:
full_test = pd.read_csv(DATA_DIR/ "derived" / "indexes" / "full_test_index.csv")
images_per_patient_ff = full_test.groupby("patient_id").size()
images_per_patient_ff.describe()

count    690.000000
mean      15.915942
std       16.735157
min        3.000000
25%        6.000000
50%       10.000000
75%       21.000000
max      240.000000
dtype: float64

In [7]:
subset_test = pd.read_csv(DATA_DIR/ "derived" / "indexes" / "subset_test_index.csv")
images_per_patient_sf = subset_test.groupby("patient_id").size()
images_per_patient_sf.describe()

count    106.000000
mean      18.754717
std       21.338903
min        3.000000
25%        5.000000
50%       11.500000
75%       22.000000
max      114.000000
dtype: float64

In [8]:
# check overlap
print("Overlap for subset?", len(set(subset_test["patient_id"]) 
                                & set(subset_train["patient_id"]))
                                )
print("Overlap for full set?", len(set(full_test["patient_id"]) & set(full_train["patient_id"])))

Overlap for subset? 0
Overlap for full set? 0


In [9]:
all_patients = (full_train.groupby("patient_id", as_index = False).agg(
    patient_target = ('target','max'),
    image_count = ('image_name', 'size'),
    melanoma_count = ("target", 'sum')))

In [10]:
all_patients

,patient_id,patient_target,image_count,melanoma_count
0,IP_0001230,0,6,0
1,IP_0019713,0,13,0
2,IP_0036322,0,5,0
3,IP_0038436,0,3,0
4,IP_0038545,1,28,1
...,...,...,...,...
2051,IP_9989332,0,10,0
2052,IP_9992027,0,39,0
2053,IP_9995095,0,14,0
2054,IP_9996429,0,3,0


In [11]:
all_patients.dtypes

patient_id          str
patient_target    int64
image_count       int64
melanoma_count    int64
dtype: object

In [12]:
all_patients["patient_target"].value_counts()


patient_target
0    1628
1     428
Name: count, dtype: int64

In [13]:
train_patients, temp = train_test_split(
    all_patients,
    test_size = 0.2,
    stratify=all_patients['patient_target'],
    random_state=42)

In [14]:
validation_patients, test_patients = train_test_split(
    temp,
    test_size = 0.5,
    stratify = temp['patient_target'],
    random_state = 42)

train_patients = train_patients.assign(model_split='train')
validation_patients = validation_patients.assign(model_split='validation')
test_patients = test_patients.assign(model_split='test')

In [15]:
train_patients

,patient_id,patient_target,image_count,melanoma_count,model_split
1869,IP_9164625,0,23,0,train
1418,IP_7123416,0,3,0,train
649,IP_3260326,0,3,0,train
1342,IP_6730678,0,17,0,train
713,IP_3562983,0,71,0,train
...,...,...,...,...,...
1758,IP_8663649,0,61,0,train
1554,IP_7748972,0,16,0,train
11,IP_0063457,0,11,0,train
1476,IP_7404837,0,6,0,train


In [16]:
patient_assignments = pd.concat(
    [train_patients, validation_patients,test_patients],
    ignore_index=True)

In [17]:
patient_assignments['patient_id'].is_unique

True

In [18]:
full_train_split_index = full_train.merge(
    patient_assignments[["patient_id", "model_split"]],
    on= "patient_id",
    how = 'left',
    validate = 'many_to_one'
)

In [19]:
full_train_split_index

,image_name,patient_id,sex,age_approx,anatom_site_general_challenge,diagnosis,benign_malignant,target,jpeg_path,dcm_path,dataset_scope,data_split,model_split
0,ISIC_2637011,IP_7279968,male,45.0,head/neck,unknown,benign,0,data/jpeg/train/ISIC_2637011.jpg,data/dicom/train/ISIC_2637011.dcm,full,train,test
1,ISIC_0015719,IP_3075186,female,45.0,upper extremity,unknown,benign,0,data/jpeg/train/ISIC_0015719.jpg,data/dicom/train/ISIC_0015719.dcm,full,train,train
2,ISIC_0052212,IP_2842074,female,50.0,lower extremity,nevus,benign,0,data/jpeg/train/ISIC_0052212.jpg,data/dicom/train/ISIC_0052212.dcm,full,train,train
3,ISIC_0068279,IP_6890425,female,45.0,head/neck,unknown,benign,0,data/jpeg/train/ISIC_0068279.jpg,data/dicom/train/ISIC_0068279.dcm,full,train,train
4,ISIC_0074268,IP_8723313,female,55.0,upper extremity,unknown,benign,0,data/jpeg/train/ISIC_0074268.jpg,data/dicom/train/ISIC_0074268.dcm,full,train,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...
33121,ISIC_9999134,IP_6526534,male,50.0,torso,unknown,benign,0,data/jpeg/train/ISIC_9999134.jpg,data/dicom/train/ISIC_9999134.dcm,full,train,train
33122,ISIC_9999320,IP_3650745,male,65.0,torso,unknown,benign,0,data/jpeg/train/ISIC_9999320.jpg,data/dicom/train/ISIC_9999320.dcm,full,train,train
33123,ISIC_9999515,IP_2026598,male,20.0,lower extremity,unknown,benign,0,data/jpeg/train/ISIC_9999515.jpg,data/dicom/train/ISIC_9999515.dcm,full,train,train
33124,ISIC_9999666,IP_7702038,male,50.0,lower extremity,unknown,benign,0,data/jpeg/train/ISIC_9999666.jpg,data/dicom/train/ISIC_9999666.dcm,full,train,validation


In [20]:
full_train_split_index['model_split'].notna().all()

np.True_

### Check split melanoma distributions for images and patients

In [21]:
full_train_split_index.groupby("model_split").agg(
    images = ('image_name','size'),
    patients = ('patient_id', 'nunique'),
    melanoma_images = ('target', 'sum'),
    melanoma_rate = ('target', "mean")
)

,images,patients,melanoma_images,melanoma_rate
model_split,,,,
test,3390,206,53,0.015634
train,26755,1644,475,0.017754
validation,2981,206,56,0.018786


In [22]:
patient_assignments.groupby("model_split")['patient_target'].agg(['count','sum','mean'])

,count,sum,mean
model_split,,,
test,206,43,0.208738
train,1644,342,0.208029
validation,206,43,0.208738


In [23]:
# save csv file
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


full_train_split_index.to_csv(OUTPUT_DIR/"full_train_split_index.csv", index=False)
patient_assignments.to_csv(OUTPUT_DIR/"patient_assignments.csv")

In [24]:
subset_index = pd.concat([subset_train,subset_test], ignore_index = True)



In [26]:
subset_patient_ids = subset_index["patient_id"].unique()

patient_assignments[
    patient_assignments["patient_id"].isin(subset_patient_ids)
].groupby("model_split")["patient_id"].nunique()

model_split
train    528
Name: patient_id, dtype: int64

In [27]:
patient_assignments["model_split"].value_counts()

model_split
train         1644
validation     206
test           206
Name: count, dtype: int64

In [28]:
subset_train = subset_train.assign(data_split='train')
subset_test = subset_test.assign(data_split='test')


In [29]:
subset_train_patients = (
    subset_train.groupby('patient_id',as_index=False)
    .agg(
        patient_target=('target','max'),
        image_count=('image_name','size'),
        melanoma_images=('target','sum')
    )
)


subset_test_patients = (
    subset_test.groupby('patient_id',as_index=False)
    .agg(
        patient_target=('target','max'),
        image_count=('image_name','size'),
        melanoma_images=('target','sum')
    )
)

In [30]:
subset_model_train_patients, subset_validation_patients = train_test_split(subset_train_patients, test_size=0.15,stratify=subset_train_patients['patient_target'], random_state=42)

In [31]:
subset_model_train_patients = subset_model_train_patients.assign(model_split='train')
subset_validation_patients = subset_validation_patients.assign(model_split='validation')
subset_test_patients = subset_test_patients.assign(model_split='test')

subset_patient_assignments = pd.concat([subset_model_train_patients,subset_validation_patients,subset_test_patients], )

subset_split_index = subset_index.merge(patient_assignments[['patient_id','model_split']], on='patient_id',how='left',validate='many_to_one')


In [32]:
subset_patient_assignments.groupby("model_split").agg(
    patients=("patient_id", "size"),
    positive_patients=("patient_target", "sum"),
    positive_patient_rate=("patient_target", "mean")
)

,patients,positive_patients,positive_patient_rate
model_split,,,
test,106,22,0.207547
train,358,75,0.209497
validation,64,13,0.203125


In [33]:
subset_patient_assignments['model_split'].value_counts()

model_split
train         358
test          106
validation     64
Name: count, dtype: int64

In [34]:
subset_split_index['data_split'].value_counts()

data_split
train    6864
test     1988
Name: count, dtype: int64

In [35]:
print(
    "Train assignments:",
    subset_model_train_patients["model_split"].value_counts()
)

print(
    "Validation assignments:",
    subset_validation_patients["model_split"].value_counts()
)

print(
    "Test assignments:",
    subset_test_patients["model_split"].value_counts()
)

Train assignments: model_split
train    358
Name: count, dtype: int64
Validation assignments: model_split
validation    64
Name: count, dtype: int64
Test assignments: model_split
test    106
Name: count, dtype: int64


In [36]:
subset_split_index.groupby("model_split").agg(
    images=("image_name", "size"),
    patients=("patient_id", "nunique"),
    melanoma_images=("target", "sum"),
    melanoma_rate=("target", "mean")
)

,images,patients,melanoma_images,melanoma_rate
model_split,,,,
train,8852,528,159,0.017962


In [37]:
subset_split_index = subset_split_index.drop(
    columns="model_split",
    errors="ignore"
)

subset_split_index = subset_split_index.merge(
    subset_patient_assignments[["patient_id", "model_split"]],
    on="patient_id",
    how="left",
    validate="many_to_one"
)

In [38]:
subset_split_index

,image_name,patient_id,sex,age_approx,anatom_site_general_challenge,diagnosis,benign_malignant,target,jpeg_path,dcm_path,dataset_scope,data_split,model_split
0,ISIC_0015719,IP_3075186,female,45.0,upper extremity,unknown,benign,0,data/subset/jpeg/train/ISIC_0015719.jpg,data/dicom/train/ISIC_0015719.dcm,subset,train,train
1,ISIC_0074542,IP_4698288,male,25.0,lower extremity,unknown,benign,0,data/subset/jpeg/train/ISIC_0074542.jpg,data/dicom/train/ISIC_0074542.dcm,subset,train,train
2,ISIC_0076545,IP_9802602,male,55.0,upper extremity,unknown,benign,0,data/subset/jpeg/train/ISIC_0076545.jpg,data/dicom/train/ISIC_0076545.dcm,subset,train,train
3,ISIC_0082348,IP_7684360,male,55.0,torso,unknown,benign,0,data/subset/jpeg/train/ISIC_0082348.jpg,data/dicom/train/ISIC_0082348.dcm,subset,train,validation
4,ISIC_0085172,IP_1705144,female,50.0,lower extremity,unknown,benign,0,data/subset/jpeg/train/ISIC_0085172.jpg,data/dicom/train/ISIC_0085172.dcm,subset,train,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8847,ISIC_9981394,IP_8352243,female,40.0,upper extremity,unknown,benign,0,data/subset/jpeg/test/ISIC_9981394.jpg,data/dicom/train/ISIC_9981394.dcm,subset,test,test
8848,ISIC_9998152,IP_4356379,female,40.0,lower extremity,unknown,benign,0,data/subset/jpeg/test/ISIC_9998152.jpg,data/dicom/train/ISIC_9998152.dcm,subset,test,test
8849,ISIC_9998682,IP_2516168,male,60.0,head/neck,melanoma,malignant,1,data/subset/jpeg/test/ISIC_9998682.jpg,data/dicom/train/ISIC_9998682.dcm,subset,test,test
8850,ISIC_9999127,IP_9583707,male,20.0,torso,unknown,benign,0,data/subset/jpeg/test/ISIC_9999127.jpg,data/dicom/train/ISIC_9999127.dcm,subset,test,test


In [39]:

subset_split_index.to_csv(OUTPUT_DIR/"subset_split_index.csv", index=False)
subset_patient_assignments.to_csv(OUTPUT_DIR/"subset_patient_assignments.csv", index=False)

In [40]:
subset_split_index.groupby("model_split").agg(
    images=("image_name", "size"),
    patients=("patient_id", "nunique"),
    melanoma_images=("target", "sum"),
    melanoma_rate=("target", "mean")
)

,images,patients,melanoma_images,melanoma_rate
model_split,,,,
test,1988,106,31,0.015594
train,5740,358,110,0.019164
validation,1124,64,18,0.016014


The outputs of this notebook were:
- full_train_split_index.csv
- patient_assignments.csv
- subset_split_index.csv
- subset_patient_assignments.csv


The subset training dataset was split into 'training' (85%) and 'testing' (15%) datasets, which when combined with the subset testing dataset, the subset_split_index now has an approximate model split of train - 65%, validation - 13%, and test - 22%.